# CEMS connection

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import duckdb

In [4]:
# In-memory db
con = duckdb.connect()

In [5]:
con.install_extension("httpfs")
con.load_extension("httpfs")

There are two API endpoints for two different services offered by the **Emergency Management Service**:

- **Mapping API**: contains all the active and past mapping activations.
- **Rapid Mapping API**: contains information about emergency mapping activations performed by the Copernicu Rapid Mapping management team. This is a subset of the activations exposed by the previous endpoint.

In [6]:
url_mapping = "https://mapping.emergency.copernicus.eu/activations/api/activations/"
url_rapidmapping = "https://rapidmapping.emergency.copernicus.eu/backend/dashboard-api/public-activations-info/"
url_extended = "https://rapidmapping.emergency.copernicus.eu/backend/dashboard-api/public-activations/"

In [7]:
df_mapping = con.execute(f"SELECT * FROM read_json_auto('{url_mapping}')").df()
df_mapping

,count,next,previous,results
0,1041,http://mapping.emergency.copernicus.eu/activat...,None,"[{'code': 'EMSR908', 'countries': [{'short_nam..."


The `next` field returns the endpoint to hit with the paginated query applied for the next batch of data.

In [15]:
df_mapping["next"].item()

'http://mapping.emergency.copernicus.eu/activations/api/activations/?limit=20&offset=20'

In [16]:
df_rapidmapping = con.execute(f"SELECT * FROM read_json_auto('{url_rapidmapping}')").df()
df_rapidmapping

,count,next,previous,results
0,243,https://rapidmapping.emergency.copernicus.eu/b...,None,"[{'code': 'EMSR908', 'countries': ['Spain', 'P..."


In [37]:
df_extended = con.execute(f"SELECT * FROM read_json_auto('{url_extended}')").df()
df_extended

,code
0,Activation code parameter is mandatory


With DuckDB we can also unnest directly the `results` field:

In [64]:
df = con.execute(f"SELECT unnest(results) as 'Activation' FROM read_json_auto('{url_mapping}')").df()
df

,Activation
0,"{'code': 'EMSR908', 'countries': [{'short_name..."
1,"{'code': 'EMSR907', 'countries': [{'short_name..."
2,"{'code': 'EMSR906', 'countries': [{'short_name..."
3,"{'code': 'EMSR905', 'countries': [{'short_name..."
4,"{'code': 'EMSR904', 'countries': [{'short_name..."
5,"{'code': 'EMSR903', 'countries': [{'short_name..."
6,"{'code': 'EMSR902', 'countries': [{'short_name..."
7,"{'code': 'EMSR901', 'countries': [{'short_name..."
8,"{'code': 'EMSR900', 'countries': [{'short_name..."
9,"{'code': 'EMSR899', 'countries': [{'short_name..."


In [65]:
df.iloc[7].to_dict()

{'Activation': {'code': 'EMSR901',
  'countries': [{'short_name': 'Germany'}],
  'category': {'slug': 'fire', 'name': 'Wildfire'},
  'name': 'Wildfire in Primstal, Germany',
  'centroid': 'POINT (6.948453277542661 49.54527245112298)',
  'activationTime': datetime.datetime(2026, 7, 23, 19, 16),
  'lastUpdate': '2026-07-24T21:20:09.141863',
  'drmPhase': 'response',
  'closed': True,
  'n_aois': 1,
  'n_products': 1,
  'search_snippet': 'On the 23 July 2026 at 17:30 UTC, a wildfire is reported to have affected the northern Saarland region, and spread up the hillside. The events are on-going, with damage reported to affect forestry, protected natural areas, infrastructure, road networks, power supply and water supply, while evacua...',
  'past_items_search_text': ''}}

In [26]:
code = "EMSR908"
url_cems = url_rapidmapping + f"?code={code}"
df_908 = con.execute(f"SELECT * FROM read_json_auto('{url_cems}')").df()

In [27]:
df_908["results"].to_dict()

{0: array([{'code': 'EMSR908', 'countries': ['Spain', 'Portugal'], 'eventTime': datetime.datetime(2026, 7, 28, 23, 7), 'name': 'Wildfires in Castilla y Leon, Spain and Norte, Portugal', 'centroid': 'POINT (-6.29 41.375)', 'activationTime': datetime.datetime(2026, 7, 30, 10, 51), 'category': 'Wildfire', 'lastUpdate': '2026-07-30T14:13:23.641004', 'closed': False, 'gdacsId': None, 'n_aois': 1, 'n_products': 1}],
       dtype=object)}

In [28]:
code = "EMSR901"
url_cems = url_rapidmapping + f"?code={code}"
df_901 = con.execute(f"SELECT * FROM read_json_auto('{url_cems}')").df()

In [29]:
df_901["results"].to_dict()

{0: array([{'code': 'EMSR901', 'countries': ['Germany'], 'eventTime': datetime.datetime(2026, 7, 23, 15, 30), 'name': 'Wildfire in Primstal, Germany', 'centroid': 'POINT (6.948453277542661 49.54527245112298)', 'activationTime': datetime.datetime(2026, 7, 23, 19, 16), 'category': 'Wildfire', 'lastUpdate': '2026-07-28T09:10:56.952459', 'closed': True, 'gdacsId': None, 'n_aois': 1, 'n_products': 1}],
       dtype=object)}

In [8]:
code = "EMSR897"
url_cems = url_extended + f"?code={code}"
df_900 = con.execute(f"SELECT * FROM read_json_auto('{url_cems}')").df()

In [9]:
dict_900 = df_900["results"].to_dict()
dict_900

{0: array([{'code': 'EMSR897', 'name': 'Wildfire in Tintwistle, United Kingdom', 'reason': 'On 16 July at 23:00, a moorland wildfire was reported to have affected Tintwistle Moor, Derbyshire, United Kingdom. The event intensified during its progression, posing a risk of further spread and of merging with other wildfires in the surrounding area, potentially affecting moorland vegetation and nearby infrastructure. Copernicus EMS Rapid Mapping was requested to provide wildfire extent emergency mapping in support of the monitoring and management of the event.', 'category': 'Wildfire', 'subCategory': 'Land fire: brush, bush, pasture', 'sensitive': False, 'reportLink': 'https://storymaps.arcgis.com/stories/c8dbfe3c21dc48eb8bfc44869922cea9', 'activator': 'United Kingdom|COBR Unit, Cabinet Office', 'eventTime': datetime.datetime(2026, 7, 16, 23, 0), 'activationTime': datetime.datetime(2026, 7, 17, 14, 51), 'closed': True, 'gdacsId': None, 'continent': 'Europe', 'countries': [{'name': 'United K

In [10]:
act = dict_900[0][0]
act

{'code': 'EMSR897',
 'name': 'Wildfire in Tintwistle, United Kingdom',
 'reason': 'On 16 July at 23:00, a moorland wildfire was reported to have affected Tintwistle Moor, Derbyshire, United Kingdom. The event intensified during its progression, posing a risk of further spread and of merging with other wildfires in the surrounding area, potentially affecting moorland vegetation and nearby infrastructure. Copernicus EMS Rapid Mapping was requested to provide wildfire extent emergency mapping in support of the monitoring and management of the event.',
 'category': 'Wildfire',
 'subCategory': 'Land fire: brush, bush, pasture',
 'sensitive': False,
 'reportLink': 'https://storymaps.arcgis.com/stories/c8dbfe3c21dc48eb8bfc44869922cea9',
 'activator': 'United Kingdom|COBR Unit, Cabinet Office',
 'eventTime': datetime.datetime(2026, 7, 16, 23, 0),
 'activationTime': datetime.datetime(2026, 7, 17, 14, 51),
 'closed': True,
 'gdacsId': None,
 'continent': 'Europe',
 'countries': [{'name': 'United

In [68]:
from shapely import from_wkt

In [70]:
type(from_wkt(act['extent']))

shapely.geometry.polygon.Polygon

In [73]:
act["aois"][0]

{'name': 'Primstal',
 'extent': 'POLYGON ((6.972974 49.566041, 6.97359 49.559289, 6.984124 49.539176, 6.968519 49.529935, 6.95389 49.529302, 6.901028 49.528163, 6.943063 49.565728, 6.955031 49.568166, 6.972974 49.566041))',
 'number': 1,
 'activationCode': 'EMSR901',
 'products': [{'id': 2673,
   'type': 'DEL',
   'monitoring': False,
   'monitoringNumber': 0,
   'feasible': True,
   'images': [{'uuid': UUID('a0b2121a-9b97-4dfc-a3c8-f139a4ede911'),
     'new': True,
     'sensorType': 'optical',
     'sensorName': 'Sentinel-2',
     'resolutionClass': 'HR+',
     'acquisitionTime': datetime.datetime(2026, 7, 24, 10, 27),
     'fileName': 'EMSR901_AOI01_DEL_PRODUCT_SENTINEL2_20260724_1027_ORTHO.tif'}],
   'stats': {'Land use': {'Forests ': {'unit': 'ha',
      'total': 999.2,
      'affected': 3.6}},
    'Burnt area': {'None': {'unit': 'ha', 'total': 'NA', 'affected': 3.6}},
    'Estimated population': {'None': {'unit': '', 'total': 1400}}},
   'mapsCount': 1,
   'activationCode': 'EMSR

In [72]:
act["aois"][0]["products"][1]

{'id': 2675,
 'type': 'GRA',
 'monitoring': False,
 'monitoringNumber': 0,
 'feasible': False,
 'images': [{'uuid': UUID('e7ce2817-a0e5-4c11-bace-158dfd4ba40f'),
   'new': True,
   'sensorType': 'optical',
   'sensorName': None,
   'resolutionClass': 'HR+',
   'acquisitionTime': None,
   'fileName': ''}],
 'stats': None,
 'mapsCount': 0,
 'activationCode': 'EMSR901',
 'aoiName': 'Primstal',
 'aoiNumber': 1,
 'extent': 'POLYGON ((6.943063 49.565728, 6.955031 49.568166, 6.972974 49.566041, 6.97359 49.559289, 6.984124 49.539176, 6.968519 49.529935, 6.95389 49.529302, 6.901028 49.528163, 6.943063 49.565728))',
 'expectedDelivery': None,
 'layers': [],
 'downloadPath': '',
 'version': {'uuid': UUID('3fb796e7-9ac5-4aa0-9d2d-24b3400159d1'),
  'number': 1,
  'reason': 'Product cancelled at the request of the Authorised User',
  'deliveryTime': '2026-07-24T19:46:00',
  'statusCode': 'N'}}